# Part 2.3 MCP

After this notebook you can give an agent tools from a remote MCP server, keep only the
ones it needs, and cap their output.

In [ ]:
from kea.config import MODEL, setup
from kea.notebook import ask_and_approve, final, steps

setup("kea-rebooking-local")

## See it

Passengers also ask *why*. The booking holds the cause of record, but "is something going
on at Denver?" needs live information, so the agent calls web search from
[Exa](https://exa.ai)'s remote MCP server.

<img src="../../assets/screenshots/p2-03-mcp-trace.png" alt="the trace of a cancellation question: getbooking, then websearchexa with its truncated result." width="900">

## Build it

MCP is a standard way for a server to publish tools. The client turns them into ordinary
LangChain tools. `get_tools()` is async, so these cells use `await`.

In [ ]:
import asyncio

from langchain_mcp_adapters.client import MultiServerMCPClient

from kea.rebooking.agent import cap_results

client = MultiServerMCPClient(
    {"exa": {"url": "https://mcp.exa.ai/mcp", "transport": "streamable_http"}},
    tool_interceptors=[cap_results],
)
try:
    all_tools = await asyncio.wait_for(client.get_tools(), timeout=15)
except Exception as exc:
    all_tools = []
    print(f"Could not reach Exa ({type(exc).__name__}). Continuing without web search.")

web_search = [t for t in all_tools if t.name == "web_search_exa"]
print(f"{len(all_tools)} tools on the server, keeping {[t.name for t in web_search]}")

In [ ]:
from deepagents import create_deep_agent

from kea.rebooking.tools import get_booking

mcp_agent = create_deep_agent(
    model=MODEL,
    tools=[get_booking, *web_search],
    system_prompt=(
        "You explain flight cancellations. The booking's cancellation_cause is the cause "
        "of record. Use web search only for current airport conditions, and label it as "
        "outside information."
    ),
)

try:
    result = await ask_and_approve(
        mcp_agent,
        "Why was KA 1182 to DEN cancelled? Confirmation ABC123, last name Example. "
        "Is anything going on at Denver airport?",
    )
    steps(result)
    print()
    final(result)
except Exception as exc:
    print(f"The run failed ({type(exc).__name__}). If Exa is rate-limiting, wait a minute.")

## Under the hood

- **Transports.** `streamable_http` talks to a remote server over HTTP. `stdio` starts a
  local server as a subprocess.
- **Filter the tools.** The server offers two tools and we keep one. Every tool's name,
  description, and schema goes into the prompt on every model call, so keep what the task
  needs.
- **Tool descriptions are untrusted input.** They come from someone else's server and the
  model reads them as instructions. Print what you installed, and give an agent with web
  access no tool that moves money.
- **Cap the output.** Search results can be huge, and they are untrusted too. This
  interceptor runs around every call: it asks for three results and truncates the text.

In [ ]:
import inspect

if web_search:
    print(web_search[0].description[:300])
    print()
print(inspect.getsource(cap_results))

`load_exa_tools` in `kea/rebooking/agent.py` wraps the same client in a timeout and a
`try`/`except`, so the deployed agent builds without web search when Exa is unreachable.
Do not make it raise.

Docs: [MCP in LangChain](https://docs.langchain.com/oss/python/langchain/mcp).

## Recap

- An MCP client turns a remote server's tools into LangChain tools.
- Keep only the tools the task needs, and cap what they return.
- Treat tool descriptions and results as untrusted, and make the agent fail soft when the
  server is down.

**Next:** [Part 2.4 Backends](04_backends.ipynb)